# Llama 3.1 8B - Zero-shot ABSA Evaluation

Đánh giá khả năng zero-shot của Llama 3.1 8B Instruct trên bài toán Multilingual ABSA (EN, VI, ZH).

**Model**: NousResearch/Meta-Llama-3.1-8B-Instruct | **Quantization**: 4-bit NF4 | **Dữ liệu**: toàn bộ tập M-ABSA

## 1. Cài đặt thư viện

In [ ]:
pip install -U bitsandbytes>=0.46.1

## 2. Import thư viện

In [ ]:
import pandas as pd
import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import json
import ast
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
from collections import defaultdict

## 3. Tải mô hình

In [ ]:
model_id = "NousResearch/Meta-Llama-3.1-8B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.padding_side = "left"

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation="sdpa",
)
model.eval()
print("✅ Tải mô hình hoàn tất!")

## 4. Đọc dữ liệu M-ABSA

In [ ]:
input_dir = '/kaggle/input'
target_langs = ['en', 'vi', 'zh']
target_file = 'clean_all.txt'
all_data = []

print(f"Đang tìm kiếm file {target_file} thuộc thư mục: {target_langs}...")

for root, dirs, files in os.walk(input_dir):
    folder_name = os.path.basename(root)

    if folder_name in target_langs:
        for file in files:
            if file == target_file:
                file_path = os.path.join(root, file)
                print(f"Đang đọc và tách nhãn: {folder_name}/{file}")

                with open(file_path, 'r', encoding='utf-8') as f:
                    lines = f.readlines()

                for line in lines:
                    clean_line = line.strip()
                    if clean_line and "####" in clean_line:
                        parts = clean_line.split("####")
                        text_part = parts[0].strip()
                        labels_part = parts[1].strip()

                        try:
                            true_labels = ast.literal_eval(labels_part)
                        except:
                            true_labels = []

                        all_data.append({
                            'language': folder_name,
                            'file_name': file,
                            'text': text_part,
                            'true_labels': true_labels,
                            'domain': os.path.basename(os.path.dirname(root))
                        })

df = pd.DataFrame(all_data)
print("Số mẫu đọc được:", len(all_data))
print(df.columns)

## 5. Tiền xử lý nhãn & Categories

In [ ]:
def normalize_category(cat):
    cat = str(cat).lower().strip()
    if '#' in cat:
        return cat

    parts = cat.rsplit(' ', 1)
    if len(parts) == 2:
        aspect    = parts[0].strip().replace(' ', '_')
        attribute = parts[1].strip().replace(' ', '_')
        return f"{aspect}#{attribute}"

    return cat
def convert_to_dict_format(true_labels):
    formatted_labels = []
    for t in true_labels:
        if len(t) == 3:
            ent = str(t[0]).strip()
            cat = str(t[1]).lower().strip()
            sent = str(t[2]).lower().strip()

            formatted_labels.append({
                "entity": ent,
                "category": normalize_category(cat),
                "sentiment": sent
            })
    return formatted_labels
df['formatted_true_labels'] = df['true_labels'].apply(convert_to_dict_format)

domain_to_cats = {}
for domain, group in df.groupby('domain'):
    cats = set()
    for labels in group['formatted_true_labels']:
        for lb in labels:
            cats.add(lb['category'])
    domain_to_cats[domain] = list(cats)

# Kiểm tra
for domain, cats in domain_to_cats.items():
    print(f"{domain}: {len(cats)} categories")

unique_categories = set()
for labels in df['formatted_true_labels']:
    for label in labels:
        unique_categories.add(label['category'])

my_categories = list(unique_categories)
print(f"Đã tìm thấy {len(my_categories)} Categories thực tế trong dữ liệu: \n{my_categories}\n")

texts_to_process = df['text'].tolist()
langs = df['language'].tolist()
true_labels_list = df['formatted_true_labels'].tolist()
domains = df['domain'].tolist()

## 6. Thiết kế Prompt ABSA

In [ ]:
SYSTEM_PROMPT = (
    "You are an expert in multilingual Aspect-Based Sentiment Analysis. "
    "You ALWAYS return a valid JSON array and nothing else. "
    "No explanation, no markdown, no ```json fences."
)

FEW_SHOT = {
    "zh": """\
Example 1 (nhiều entity, giữ nguyên tiếng Trung):
Text: "服务态度很好，但是价格偏高，环境一般。"
Output: [{"entity": "服务", "category": "support#quality", "sentiment": "positive"},
         {"entity": "价格", "category": "support#price", "sentiment": "negative"},
         {"entity": "环境", "category": "support#general", "sentiment": "neutral"}]

Example 2 (NULL — không có từ entity cụ thể):
Text: "总体体验不错，就是有点贵。"
Output: [{"entity": "NULL", "category": "support#general", "sentiment": "positive"},
         {"entity": "NULL", "category": "support#price", "sentiment": "negative"}]""",

    "vi": """\
Example 1 (nhiều entity, giữ nguyên tiếng Việt):
Text: "Nhân viên phục vụ rất nhiệt tình nhưng giá hơi cao, chỗ ngồi chật."
Output: [{"entity": "nhân viên", "category": "support#quality", "sentiment": "positive"},
         {"entity": "giá", "category": "support#price", "sentiment": "negative"},
         {"entity": "chỗ ngồi", "category": "support#design_features", "sentiment": "negative"}]

Example 2 (NULL — không có từ entity cụ thể):
Text: "Nhìn chung khá ổn nhưng hơi đắt so với chất lượng."
Output: [{"entity": "NULL", "category": "support#general", "sentiment": "positive"},
         {"entity": "NULL", "category": "support#price", "sentiment": "negative"}]""",

    "en": """\
Example 1 (multiple entities):
Text: "The staff was very friendly but the price was too high and the seating was cramped."
Output: [{"entity": "staff", "category": "support#quality", "sentiment": "positive"},
         {"entity": "price", "category": "support#price", "sentiment": "negative"},
         {"entity": "seating", "category": "support#design_features", "sentiment": "negative"}]

Example 2 (NULL — no explicit entity word):
Text: "Overall decent experience but a bit overpriced for what you get."
Output: [{"entity": "NULL", "category": "support#general", "sentiment": "positive"},
         {"entity": "NULL", "category": "support#price", "sentiment": "negative"}]"""
}

def group_categories(category_list):
    groups = defaultdict(list)
    for cat in category_list:
        if '#' not in cat:
            continue
        asp, att = cat.split('#', 1)
        groups[asp].append(att)
    return "\n".join(
        f"  {asp}: [{', '.join(sorted(atts))}]"
        for asp, atts in sorted(groups.items())
    )

def build_user_prompt(text, category_list, lang="en"):
    cat_block = group_categories(category_list)

    lang_rule = {
        "zh": "Extract entity EXACTLY as it appears in the Chinese text. Do NOT translate.",
        "vi": "Extract entity EXACTLY as it appears in the Vietnamese text. Do NOT translate.",
        "en": ""
    }.get(lang, "")

    null_rule = """\
NULL rule:
  - Use NULL when the aspect is IMPLIED but NO specific word names it in the text.
    e.g. "This is way too expensive" → entity=NULL
  - Use the ACTUAL WORD when the entity is explicitly named in the text.
    e.g. "The service was slow" → entity="service" / "服务" / "dịch vụ" """

    example = FEW_SHOT.get(lang, FEW_SHOT["en"])

    return f"""Perform Aspect-Based Sentiment Analysis (ABSA) on the review below.

For each aspect mentioned, extract:
  - entity    : exact word/phrase from the text, or NULL if not explicitly stated
  - category  : pick ONE from the list below (format: aspect#attribute)
  - sentiment : positive | negative | neutral

Available categories:
{cat_block}

{null_rule}

{lang_rule}

Rules:
  1. One JSON object per (entity, category) pair.
  2. A sentence can have MULTIPLE entities — find ALL of them.
  3. Choose the category that best fits — do NOT invent new categories.
  4. Return [] if nothing found.
  5. Return ONLY a raw JSON array. No explanation, no markdown.

{example}

Text: "{text}"
Output:"""

## 7. Hàm Inference theo Batch

In [ ]:
def batch_extract_absa(texts, langs, domains, category_list, domain_to_cats,
                       batch_size=8):
    prompts = []
    for text, lang, domain in zip(texts, langs, domains):
        cats = domain_to_cats.get(domain, category_list)
        user_content = SYSTEM_PROMPT + "\n\n" + build_user_prompt(text, cats, lang)
        messages = [{"role": "user", "content": user_content}]
        prompts.append(
            tokenizer.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True
            )
        )

    encoded_lengths = [
        len(tokenizer.encode(p, add_special_tokens=False))
        for p in prompts
    ]

    sorted_indices = sorted(range(len(prompts)), key=lambda i: encoded_lengths[i])
    sorted_prompts = [prompts[i] for i in sorted_indices]

    all_responses_sorted = []

    for i in tqdm(range(0, len(sorted_prompts), batch_size),
                  desc="Tiến độ phân tích ABSA"):
        batch = sorted_prompts[i : i + batch_size]

        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=1536,
        ).to(model.device)

        with torch.no_grad():
            generated = model.generate(
                **inputs,
                max_new_tokens=150,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
                use_cache=True,
                repetition_penalty=1.1,
            )

        new_tokens = [
            out[len(inp):]
            for inp, out in zip(inputs.input_ids, generated)
        ]
        responses = tokenizer.batch_decode(new_tokens, skip_special_tokens=True)
        all_responses_sorted.extend(responses)

        del inputs, generated, new_tokens
        torch.cuda.empty_cache()

    all_responses = [None] * len(prompts)
    for original_idx, sorted_pos in enumerate(sorted_indices):
        all_responses[sorted_pos] = all_responses_sorted[original_idx]

    return all_responses

## 8. Hàm tính Metrics

In [ ]:
def calculate_metrics(true_list, pred_list):
    true_set = set()
    for t in true_list:
        ent = str(t.get('entity', '')).lower().strip()
        cat = str(t.get('category', '')).lower().strip()
        sent = str(t.get('sentiment', '')).lower().strip()
        if ent or cat or sent:
            true_set.add((ent, cat, sent))

    pred_set = set()
    if isinstance(pred_list, list):
        for p in pred_list:
            if isinstance(p, dict):
                ent = str(p.get('entity', '')).lower().strip()
                cat = str(p.get('category', '')).lower().strip()
                sent = str(p.get('sentiment', '')).lower().strip()
                if ent or cat or sent:
                    pred_set.add((ent, cat, sent))

    tp = len(true_set.intersection(pred_set))
    fp = len(pred_set - true_set)
    fn = len(true_set - pred_set)
    return tp, fp, fn

## 9. Chạy dự đoán & Đánh giá

In [ ]:
torch.cuda.empty_cache()
print(f"Bắt đầu chạy dự đoán trên {len(df)} mẫu dữ liệu...")

BATCH_SIZE = 8

raw_outputs = batch_extract_absa(
    texts_to_process,
    langs,
    domains,
    my_categories,
    domain_to_cats,
    batch_size=BATCH_SIZE
)
metrics_by_lang = {lang: {'tp': 0, 'fp': 0, 'fn': 0} for lang in df['language'].unique()}
comparison_data = []

for i, raw_output in enumerate(raw_outputs):
    try:
        clean_json = raw_output.replace("```json", "").replace("```", "").strip()
        parsed_json = json.loads(clean_json)

        if isinstance(parsed_json, dict):
            for key, val in parsed_json.items():
                if isinstance(val, list):
                    parsed_json = val
                    break
        if not isinstance(parsed_json, list):
            parsed_json = []

    except json.JSONDecodeError:
        parsed_json = []

    lang = langs[i]
    tp, fp, fn = calculate_metrics(true_labels_list[i], parsed_json)

    metrics_by_lang[lang]['tp'] += tp
    metrics_by_lang[lang]['fp'] += fp
    metrics_by_lang[lang]['fn'] += fn

    comparison_data.append({
        'Language': lang,
        'Text': texts_to_process[i],
        'True_Labels': json.dumps(true_labels_list[i], ensure_ascii=False),
        'Predicted_Labels': json.dumps(parsed_json, ensure_ascii=False)
    })

df_comparison = pd.DataFrame(comparison_data)
output_file_csv = 'absa_comparison_results.csv'
df_comparison.to_csv(output_file_csv, index=False, encoding='utf-8-sig')
print(f"\nĐã lưu file: {output_file_csv}")

## 10. Tổng hợp kết quả F1-Score

In [ ]:
f1_scores = {}
print("\n--- KẾT QUẢ ĐÁNH GIÁ (F1-SCORE) ---")
for lang, counts in metrics_by_lang.items():
    tp = counts['tp']
    fp = counts['fp']
    fn = counts['fn']

    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    f1_scores[lang] = f1
    print(f"Ngôn ngữ: {lang.upper()} | Precision: {precision:.2f} | Recall: {recall:.2f} | F1: {f1:.2f}")

## 11. Biểu đồ kết quả

In [ ]:
plot_langs = list(f1_scores.keys())
plot_scores = list(f1_scores.values())

plt.figure(figsize=(8, 5))
plt.plot(plot_langs, plot_scores, marker='o', linestyle='-', color='#1f77b4', linewidth=2, markersize=8)
plt.title('So sánh điểm F1-Score ABSA giữa các ngôn ngữ', fontsize=14, pad=15)
plt.xlabel('Ngôn ngữ', fontsize=12)
plt.ylabel('F1 Score', fontsize=12)
plt.ylim(0, 1.1)
plt.grid(True, linestyle='--', alpha=0.6)

for i, score in enumerate(plot_scores):
    plt.text(plot_langs[i], score + 0.05, f"{score:.2f}", ha='center', fontweight='bold', fontsize=11)

plt.show()